In [5]:
import pandas as pd

train_raw = pd.read_csv("../data/train.tsv", sep="\t")

print("Shape:", train_raw.shape)
print(train_raw.head())
print(train_raw.columns.tolist())

Shape: (1482535, 8)
   train_id                                 name  item_condition_id  \
0         0  MLB Cincinnati Reds T Shirt Size XL                  3   
1         1     Razer BlackWidow Chroma Keyboard                  3   
2         2                       AVA-VIV Blouse                  1   
3         3                Leather Horse Statues                  1   
4         4                 24K GOLD plated rose                  1   

                                       category_name brand_name  price  \
0                                  Men/Tops/T-shirts        NaN   10.0   
1  Electronics/Computers & Tablets/Components & P...      Razer   52.0   
2                        Women/Tops & Blouses/Blouse     Target   10.0   
3                 Home/Home Décor/Home Décor Accents        NaN   35.0   
4                            Women/Jewelry/Necklaces        NaN   44.0   

   shipping                                   item_description  
0         1                                

In [6]:
print("Raw shape:", train_raw.shape)
print("Processed shape:", train_df.shape)

Raw shape: (1482535, 8)
Processed shape: (1185328, 28)


# 1. Dataset Loading and Initial Validation

## 1.1 Raw Dataset

The original Mercari training dataset is stored in `data/train.tsv`. It is a tab-separated values (TSV) file containing the following eight columns:

- `train_id`: Unique identifier for each product listing.
- `name`: Product name.
- `item_condition_id`: Condition of the product.
- `category_name`: Product category hierarchy.
- `brand_name`: Product brand.
- `price`: Product listing price and prediction target.
- `shipping`: Indicates the shipping arrangement.
- `item_description`: Description of the product.

The raw dataset was loaded using pandas with the tab character specified as the delimiter.

**Raw dataset dimensions:** 1,482,535 rows × 8 columns.

## 1.2 Processed Dataset

The existing processed training dataset contains 28 columns, including the original product attributes and engineered features.

Examples of engineered features include:

- Cleaned product names and descriptions.
- Name and description lengths and word counts.
- Hierarchical category encodings.
- Brand encoding.
- Indicators for the presence of brand information and descriptions.
- Log-transformed length and word-count features.
- `log_price`, a logarithmic transformation of the target variable.

**Processed dataset dimensions:** 1,185,328 rows × 28 columns.

The processed dataset is stored separately from the original TSV file.

## 1.3 Dataset Comparison

A comparison of the raw and processed datasets revealed a difference in their row counts.

| Property | Raw dataset | Processed dataset |
|---|---:|---:|
| Number of rows | 1,482,535 | 1,185,328 |
| Number of columns | 8 | 28 |
| Format | TSV | CSV |
| Processing status | Original data | Cleaned and feature-engineered data |

The processed dataset contains 297,207 fewer rows than the raw dataset, corresponding to approximately 20.05% of the original records.

## 1.4 Validation Requirement

The reason for this reduction has not yet been established. Potential explanations include intentional sampling, filtering, or differences between processing runs.

The record identifiers (`train_id`) will be compared between the raw and processed datasets to determine whether the processed data is a subset of the original data.

No conclusion about the cause of the missing records will be made until this comparison is complete.

## 1.5 Next Steps

1. Compare the `train_id` values in both datasets.
2. Investigate why records may have been excluded.
3. Verify that the processed data was derived from the intended raw dataset.
4. Confirm that missing values and categorical features were handled consistently.
5. Verify the preprocessing and feature-engineering pipeline before model development.

**Important:** Valid product listings should not be removed without a clear reason. Missing brand names or descriptions can often be handled through appropriate preprocessing rather than deleting entire records.

In [7]:
raw_ids = set(train_raw["train_id"])
processed_ids = set(train_df["train_id"])

print("Rows missing from processed data:",
      len(raw_ids - processed_ids))

print("Unexpected IDs in processed data:",
      len(processed_ids - raw_ids))

Rows missing from processed data: 297207
Unexpected IDs in processed data: 0


## 1.6 Record Identifier Validation

The record identifiers (`train_id`) were compared between the raw and processed training datasets to verify their relationship.

The comparison produced the following results:

| Validation check | Result |
|---|---:|
| Raw records absent from processed data | 297,207 |
| Processed records not found in raw data | 0 |
| Raw dataset size | 1,482,535 |
| Processed dataset size | 1,185,328 |

### Findings

- Every record in the processed dataset has a corresponding identifier in the raw dataset.
- The processed dataset contains a subset of the original training records.
- A total of 297,207 raw records, approximately 20.05% of the original dataset, are not present in the processed dataset.
- The identifier comparison confirms that no unexpected record identifiers were introduced during processing.

### Conclusion

The processed dataset is derived from records in the original training dataset. However, the reason for the reduction in record count remains unknown.

Further investigation is required to identify whether the reduction resulted from intentional sampling, filtering, or another preprocessing operation. The row-count discrepancy should be resolved before proceeding with final model development.



## Record Count Validation and Train–Validation Split

The raw dataset contains 1,482,535 records. During preprocessing, listings with non-positive prices are removed, and the remaining records are divided into training and validation sets using an 80/20 split with a fixed random seed of 42.

The processed training dataset contains 1,185,328 records, while the validation dataset contains 296,333 records. Together, these datasets contain 1,481,661 records, which is 874 fewer than the original dataset.

The apparent difference between the raw and training datasets is therefore primarily explained by the train–validation split, rather than by the loss of approximately 20% of the original data.

The remaining difference of 874 records should be verified against the number of listings with non-positive prices in the raw dataset.

**Conclusion:** The training and validation datasets represent separate subsets of the original data. Their identifiers should be checked together to confirm that there is no unintended overlap or unexplained record loss.

In [8]:
print("Non-positive prices in raw data:", (train_raw["price"] <= 0).sum())
print("Combined processed rows:", len(train_df) + len(valid_df))
print("Raw rows:", len(train_raw))

Non-positive prices in raw data: 874
Combined processed rows: 1481661
Raw rows: 1482535


In [9]:
train_ids = set(train_df["train_id"])
valid_ids = set(valid_df["train_id"])
raw_ids = set(train_raw.loc[train_raw["price"] > 0, "train_id"])

print("Overlap between train and validation:", len(train_ids & valid_ids))
print("Eligible raw IDs missing from both:", len(raw_ids - train_ids - valid_ids))
print("Processed IDs not in eligible raw data:", len((train_ids | valid_ids) - raw_ids))

Overlap between train and validation: 0
Eligible raw IDs missing from both: 0
Processed IDs not in eligible raw data: 0


## Train–Validation Split Verification

The record identifiers (`train_id`) were compared across the eligible raw dataset, processed training dataset, and processed validation dataset.

| Validation check | Result |
|---|---:|
| Records shared between training and validation | 0 |
| Eligible raw records missing from both datasets | 0 |
| Processed records absent from eligible raw data | 0 |

### Conclusion

The train–validation split is complete and mutually exclusive. Every raw record with a positive price appears in exactly one of the processed datasets, and no unexpected records were introduced.

The split is therefore validated with respect to record coverage and identifier overlap.

In [10]:
# Check the cleaned text features
text_columns = ["name_cleaned", "item_description_cleaned"]

for col in text_columns:
    print(f"\n{col}")
    print("Train missing values:", train_df[col].isna().sum())
    print("Validation missing values:", valid_df[col].isna().sum())
    print("Sample:", train_df[col].dropna().iloc[0][:200])


name_cleaned
Train missing values: 15
Validation missing values: 4
Sample: brand new skull candy headphones

item_description_cleaned
Train missing values: 439
Validation missing values: 129
Sample: never used


In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Replace missing text without removing any rows
for df in [train_df, valid_df]:
    df["name_cleaned"] = df["name_cleaned"].fillna("")
    df["item_description_cleaned"] = (
        df["item_description_cleaned"].fillna("")
    )

# Fit vectorizers on training data only
name_vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=2,
    dtype="float32"
)

description_vectorizer = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    dtype="float32"
)

# Learn vocabulary and transform training text
X_name_train = name_vectorizer.fit_transform(train_df["name_cleaned"])
X_desc_train = description_vectorizer.fit_transform(
    train_df["item_description_cleaned"]
)

# Transform validation text using the same learned vocabularies
X_name_valid = name_vectorizer.transform(valid_df["name_cleaned"])
X_desc_valid = description_vectorizer.transform(
    valid_df["item_description_cleaned"]
)

print("Name TF-IDF train:", X_name_train.shape)
print("Description TF-IDF train:", X_desc_train.shape)
print("Name TF-IDF validation:", X_name_valid.shape)
print("Description TF-IDF validation:", X_desc_valid.shape)

C:\jupyter\.venv\Lib\site-packages\sklearn\feature_extraction\text.py:2049: UserWarning: Only (<class 'numpy.float64'>, <class 'numpy.float32'>, <class 'numpy.float16'>) 'dtype' should be used. float32 'dtype' will be converted to np.float64.
  warnings.warn(


Name TF-IDF train: (1185328, 10000)
Description TF-IDF train: (1185328, 20000)
Name TF-IDF validation: (296333, 10000)
Description TF-IDF validation: (296333, 20000)


## 2. TF-IDF Text Feature Extraction

### 2.1 Objective

The Mercari dataset contains textual information in the product name and item description. These fields may contain useful information about product type, brand, condition, and other characteristics that influence listing prices.

Machine learning models require numerical inputs. Term Frequency–Inverse Document Frequency (TF-IDF) converts text into numerical feature vectors based on the importance of words and word combinations within the dataset.

### 2.2 Handling Missing Text

Missing values in `name_cleaned` and `item_description_cleaned` were replaced with empty strings.

| Column | Missing values in training | Missing values in validation |
|---|---:|---:|
| `name_cleaned` | 15 | 4 |
| `item_description_cleaned` | 439 | 129 |

Empty strings were used instead of removing records, preserving the verified training and validation dataset sizes.

### 2.3 TF-IDF Configuration

Two separate TF-IDF vectorizers were created.

| Parameter | Product name | Item description |
|---|---:|---:|
| Maximum features | 10,000 | 20,000 |
| N-gram range | (1, 2) | (1, 2) |
| Minimum document frequency | 2 | 2 |
| Maximum document frequency | Default | 0.95 |
| Data type | `float32` | `float32` |

Unigrams represent individual words, while bigrams represent consecutive pairs of words. Using both allows the model to capture individual terms and short phrases.

The vocabulary limits control memory usage. The description vectorizer also excludes terms appearing in more than 95% of documents.

### 2.4 Preventing Data Leakage

Each vectorizer was fitted exclusively on the training dataset using `fit_transform()`. The fitted vectorizers were then applied to the validation dataset using `transform()`.

This ensures that vocabulary selection is based only on training data. Validation data does not influence the learned vocabulary.

### 2.5 Sparse Representation

TF-IDF matrices are stored in sparse format because most documents contain only a small fraction of the vocabulary. Sparse matrices reduce memory consumption compared with dense representations.

The extracted features are stored separately for product names and item descriptions:

- `X_name_train`
- `X_desc_train`
- `X_name_valid`
- `X_desc_valid`

The resulting feature matrices will be inspected to verify their dimensions and can subsequently be reduced using Truncated Singular Value Decomposition (TruncatedSVD).

### 2.6 Limitations and Next Steps

TF-IDF captures word frequency and document-level importance but does not inherently understand semantic meaning. It also produces high-dimensional feature spaces.

The next stage will investigate dimensionality reduction using TruncatedSVD and integration of the text features with the existing numerical and categorical features for model training.

In [14]:
from sklearn.decomposition import TruncatedSVD

# Reduce product-name TF-IDF features
name_svd = TruncatedSVD(n_components=100, random_state=42)

X_name_train_svd = name_svd.fit_transform(X_name_train)
X_name_valid_svd = name_svd.transform(X_name_valid)

# Reduce item-description TF-IDF features
description_svd = TruncatedSVD(n_components=100, random_state=42)

X_desc_train_svd = description_svd.fit_transform(X_desc_train)
X_desc_valid_svd = description_svd.transform(X_desc_valid)

print("Name train:", X_name_train_svd.shape)
print("Name validation:", X_name_valid_svd.shape)
print("Description train:", X_desc_train_svd.shape)
print("Description validation:", X_desc_valid_svd.shape)

print(
    "Name variance captured:",
    name_svd.explained_variance_ratio_.sum()
)

print(
    "Description variance captured:",
    description_svd.explained_variance_ratio_.sum()
)

Name train: (1185328, 100)
Name validation: (296333, 100)
Description train: (1185328, 100)
Description validation: (296333, 100)
Name variance captured: 0.2089306
Description variance captured: 0.23083235


## 3. Dimensionality Reduction Using TruncatedSVD

### 3.1 Objective

TF-IDF generates high-dimensional, sparse feature matrices. The product-name vectorizer produces 10,000 features, while the item-description vectorizer produces 20,000 features.

To reduce computational requirements and create a more manageable feature representation, Truncated Singular Value Decomposition (TruncatedSVD) was applied separately to the two TF-IDF matrices.

### 3.2 Configuration

Two independent TruncatedSVD models were created with the following configuration:

| Parameter | Value |
|---|---|
| Product-name components | 100 |
| Item-description components | 100 |
| Random state | 42 |
| Input representation | Sparse TF-IDF matrices |

Each SVD model was fitted using only the training data. The fitted transformations were subsequently applied to the corresponding validation matrices.

### 3.3 Results

| Metric | Product name | Item description |
|---|---:|---:|
| Original TF-IDF dimensions | 10,000 | 20,000 |
| Reduced dimensions | 100 | 100 |
| Explained variance ratio sum | 20.89% | 23.08% |
| Training records | 1,185,328 | 1,185,328 |
| Validation records | 296,333 | 296,333 |

The resulting matrices have the following shapes:

- `X_name_train_svd`: `(1185328, 100)`
- `X_name_valid_svd`: `(296333, 100)`
- `X_desc_train_svd`: `(1185328, 100)`
- `X_desc_valid_svd`: `(296333, 100)`

### 3.4 Interpretation

TruncatedSVD reduced the combined text representation from 30,000 original TF-IDF dimensions to 200 components per listing.

The product-name components account for approximately 20.89% of the TF-IDF variance, while the item-description components account for approximately 23.08%.

These values describe the variance retained in the text representations. They do not directly measure predictive accuracy for product prices. The usefulness of the reduced features must therefore be evaluated during model training.

### 3.5 Feature Integration

The reduced text representations will be combined with the existing numerical and encoded categorical features.

The existing feature set contains 12 columns:

- Item condition and shipping information.
- Encoded brand and category values.
- Brand and description availability indicators.
- Log-transformed name and description length and word-count features.

Adding 100 product-name components and 100 item-description components produces a combined feature set of **212 features per listing**.

The target variable is `log_price`. The original `price`, `log_price`, and `train_id` columns are excluded from the input features to prevent target leakage and the use of record identifiers as predictors.

### 3.6 Next Steps

1. Combine the reduced text features with the existing numerical and categorical features.
2. Verify the resulting training and validation dimensions.
3. Save the fitted preprocessing components for reuse.
4. Evaluate regression models using consistent validation metrics.

The completed feature engineering pipeline provides a consistent representation of structured and textual listing information for subsequent price prediction experiments.

In [16]:
from preprocessing import FEATURE_COLUMNS, TARGET

ModuleNotFoundError: No module named 'preprocessing'

In [17]:
import os

print("Current directory:", os.getcwd())
print("Files here:", os.listdir("."))
print("Parent directory files:", os.listdir(".."))

Current directory: C:\jupyter\L-T-Project\notebooks
Files here: ['.ipynb_checkpoints', '01_EDA.ipynb', '02_Preprocessing_Feature_Engineering.ipynb']
Parent directory files: ['.git', '.gitignore', 'data', 'notebooks', 'README.md', 'src']


In [18]:
import sys
sys.path.insert(0, os.path.abspath("../src"))

from preprocessing import FEATURE_COLUMNS, TARGET

In [19]:
import numpy as np
import pandas as pd

# Existing non-text features
X_train_basic = train_df[FEATURE_COLUMNS].reset_index(drop=True)
X_valid_basic = valid_df[FEATURE_COLUMNS].reset_index(drop=True)

# Reduced text features
name_cols = [f"name_svd_{i}" for i in range(100)]
desc_cols = [f"description_svd_{i}" for i in range(100)]

X_name_train_df = pd.DataFrame(X_name_train_svd, columns=name_cols)
X_desc_train_df = pd.DataFrame(X_desc_train_svd, columns=desc_cols)

X_name_valid_df = pd.DataFrame(X_name_valid_svd, columns=name_cols)
X_desc_valid_df = pd.DataFrame(X_desc_valid_svd, columns=desc_cols)

# Combine numerical, categorical and text features
X_train = pd.concat(
    [X_train_basic, X_name_train_df, X_desc_train_df],
    axis=1
)

X_valid = pd.concat(
    [X_valid_basic, X_name_valid_df, X_desc_valid_df],
    axis=1
)

# Target: predict log_price
y_train = train_df[TARGET].to_numpy()
y_valid = valid_df[TARGET].to_numpy()

print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)
print("y_train:", y_train.shape)
print("y_valid:", y_valid.shape)

X_train: (1185328, 212)
X_valid: (296333, 212)
y_train: (1185328,)
y_valid: (296333,)


## 4. Feature Integration and Dataset Preparation

### 4.1 Objective

The numerical, encoded categorical, and reduced text features were combined into unified training and validation datasets. This provides a consistent input representation for subsequent regression model development.

### 4.2 Feature Integration

The following feature groups were combined:

| Feature group | Number of features |
|---|---:|
| Numerical and encoded categorical features | 12 |
| Reduced product-name TF-IDF features | 100 |
| Reduced item-description TF-IDF features | 100 |
| **Total** | **212** |

The original TF-IDF matrices were reduced using TruncatedSVD before integration. The resulting components were combined with the existing structured features using pandas DataFrames.

### 4.3 Dataset Dimensions

The combined datasets have the following dimensions:

| Dataset | Records | Features |
|---|---:|---:|
| Training features (`X_train`) | 1,185,328 | 212 |
| Validation features (`X_valid`) | 296,333 | 212 |
| Training target (`y_train`) | 1,185,328 | — |
| Validation target (`y_valid`) | 296,333 | — |

The feature and target dimensions match, confirming that each record has a corresponding target value.

### 4.4 Target Variable

The target variable is `log_price`, calculated using the natural logarithm of one plus the original listing price:

\[
\text{log\_price} = \ln(1+\text{price})
\]

The logarithmic transformation reduces price skewness and may improve regression model performance when prices span a wide range.

Predictions can be converted back to the original price scale using:

\[
\widehat{\text{price}} = e^{\widehat{\text{log\_price}}}-1
\]

### 4.5 Leakage Prevention

The feature matrices exclude the original `price`, transformed target `log_price`, and record identifier `train_id`.

The TF-IDF vectorizers and TruncatedSVD models were fitted on training data and applied to validation data using their fitted transformations. This prevents validation records from influencing the learned text vocabulary or dimensionality-reduction components.

### 4.6 Conclusion

Feature integration produced training and validation matrices with 212 features per record. The resulting datasets combine structured listing attributes with reduced textual representations and are ready for regression model development.

The next stage will establish a baseline model, evaluate its performance on the validation dataset, and compare alternative models using consistent evaluation metrics.


In [20]:
import os
import joblib

# Save fitted text-processing objects
os.makedirs("../data", exist_ok=True)

text_pipeline = {
    "name_vectorizer": name_vectorizer,
    "description_vectorizer": description_vectorizer,
    "name_svd": name_svd,
    "description_svd": description_svd,
}

joblib.dump(text_pipeline, "../data/text_pipeline.joblib")

print("Text pipeline saved successfully.")
print("Saved to: ../data/text_pipeline.joblib")

Text pipeline saved successfully.
Saved to: ../data/text_pipeline.joblib


In [21]:
import numpy as np

print("X_train missing values:", X_train.isna().sum().sum())
print("X_valid missing values:", X_valid.isna().sum().sum())

print("X_train infinite values:", np.isinf(X_train.to_numpy()).sum())
print("X_valid infinite values:", np.isinf(X_valid.to_numpy()).sum())

print("y_train missing values:", np.isnan(y_train).sum())
print("y_valid missing values:", np.isnan(y_valid).sum())

print("Training rows:", len(X_train) == len(y_train))
print("Validation rows:", len(X_valid) == len(y_valid))

X_train missing values: 0
X_valid missing values: 0
X_train infinite values: 0
X_valid infinite values: 0
y_train missing values: 0
y_valid missing values: 0
Training rows: True
Validation rows: True


# Notebook 02: Preprocessing and Feature Engineering

## 1. Introduction

The objective of this notebook is to prepare the Mercari Price Suggestion dataset for machine learning. The process includes validating the dataset split, handling missing values, engineering structured features, converting text into numerical representations, reducing dimensionality, and combining all features into model-ready datasets.

The target variable is `log_price`, which represents the logarithmically transformed listing price.

## 2. Dataset Validation

### 2.1 Original Dataset

The original training file, `train.tsv`, contains 1,482,535 records and eight columns:

- `train_id`
- `name`
- `item_condition_id`
- `category_name`
- `brand_name`
- `price`
- `shipping`
- `item_description`

### 2.2 Filtering Invalid Prices

Listings with non-positive prices were removed because they are unsuitable for the intended positive-price prediction task.

| Dataset validation | Result |
|---|---:|
| Original records | 1,482,535 |
| Listings with non-positive prices | 874 |
| Remaining eligible records | 1,481,661 |

The remaining records were divided into training and validation datasets using an 80/20 split with `random_state=42`.

| Dataset | Records |
|---|---:|
| Training | 1,185,328 |
| Validation | 296,333 |
| Total | 1,481,661 |

### 2.3 Train–Validation Split Verification

The `train_id` identifiers were compared across the eligible raw dataset and both processed datasets.

| Validation check | Result |
|---|---:|
| Records shared between training and validation | 0 |
| Eligible raw records missing from both datasets | 0 |
| Processed records absent from eligible raw data | 0 |

These results confirm that the split is mutually exclusive and accounts for every eligible raw record.

## 3. Existing Structured Feature Engineering

The existing preprocessing pipeline handles missing values, cleans text, extracts listing attributes, and encodes categorical information.

### 3.1 Missing-Value Handling

- Missing brand names and categories are replaced with `"unknown"`.
- Missing product names and descriptions are replaced with empty strings.
- Missing cleaned text is replaced with empty strings before TF-IDF extraction.

No records were removed because of missing text.

### 3.2 Text Cleaning

The product names and item descriptions are converted to lowercase, characters outside the English lowercase alphabet, digits, and whitespace are removed, and repeated whitespace is normalized.

The resulting columns are:

- `name_cleaned`
- `item_description_cleaned`

### 3.3 Listing-Length Features

Four features are derived from product names and descriptions:

- `name_length`
- `name_word_count`
- `description_length`
- `description_word_count`

Log-transformed versions are also generated using `log1p` to reduce skewness in the count distributions.

### 3.4 Hierarchical Category Features

The original `category_name` field contains hierarchical categories separated by `/`. It is split into three levels:

- `category_1`
- `category_2`
- `category_3`

Missing category levels are represented by `"unknown"`.

### 3.5 Brand and Description Indicators

Two binary features indicate whether a listing has a known brand and a non-empty description:

- `has_brand`
- `has_description`

Brand and category values are encoded into integer representations using label encoders fitted on the training data. Unseen validation categories and brands fall back to the `"unknown"` category.

The existing preprocessing pipeline is saved separately as `preprocessor.joblib`.

## 4. TF-IDF Text Feature Extraction

### 4.1 Objective

Product names and descriptions contain information that may help predict listing prices. TF-IDF converts text into numerical vectors based on the importance of words and word combinations.

Two independent vectorizers were created for product names and item descriptions.

### 4.2 Configuration

| Parameter | Product name | Item description |
|---|---:|---:|
| Maximum features | 10,000 | 20,000 |
| N-gram range | (1, 2) | (1, 2) |
| Minimum document frequency | 2 | 2 |
| Maximum document frequency | Default | 0.95 |
| Data type | `float32` | `float32` |

Unigrams represent individual words, while bigrams represent consecutive pairs of words.

### 4.3 TF-IDF Results

| Matrix | Shape |
|---|---|
| Training product names | `(1185328, 10000)` |
| Validation product names | `(296333, 10000)` |
| Training descriptions | `(1185328, 20000)` |
| Validation descriptions | `(296333, 20000)` |

The configured vocabulary limits were reached for both vectorizers.

### 4.4 Preventing Data Leakage

Both vectorizers were fitted exclusively on training data using `fit_transform()`. The fitted vectorizers were then applied to validation data using `transform()`.

This prevents validation data from influencing the learned vocabulary.

## 5. Dimensionality Reduction Using TruncatedSVD

### 5.1 Objective

The combined TF-IDF representation contains 30,000 potential text features per listing. Truncated Singular Value Decomposition (TruncatedSVD) was used to reduce the dimensionality of these sparse matrices.

### 5.2 Configuration

Two separate SVD models were fitted:

| Parameter | Value |
|---|---:|
| Product-name components | 100 |
| Description components | 100 |
| Random state | 42 |

Each SVD model was fitted on its corresponding training matrix and then applied to the validation matrix.

### 5.3 Results

| Metric | Product name | Item description |
|---|---:|---:|
| Original dimensions | 10,000 | 20,000 |
| Reduced dimensions | 100 | 100 |
| Explained variance ratio sum | 20.89% | 23.08% |

The combined text representation is reduced from 30,000 dimensions to 200 components per listing.

The explained variance figures describe the variance retained in the respective TF-IDF representations. They do not directly measure the predictive information retained for price estimation.

## 6. Feature Integration

### 6.1 Feature Groups

The reduced text components were combined with the existing numerical and encoded categorical features.

| Feature group | Number of features |
|---|---:|
| Numerical and encoded categorical features | 12 |
| Product-name SVD components | 100 |
| Description SVD components | 100 |
| **Total** | **212** |

The original `price`, `log_price`, and `train_id` columns are excluded from the input feature matrix.

### 6.2 Final Dataset Dimensions

| Dataset | Records | Features |
|---|---:|---:|
| `X_train` | 1,185,328 | 212 |
| `X_valid` | 296,333 | 212 |
| `y_train` | 1,185,328 | — |
| `y_valid` | 296,333 | — |

The training and validation feature matrices have matching column structures and the expected number of records.

### 6.3 Target Transformation

The target variable is calculated as:

\[
\text{log\_price}=\ln(1+\text{price})
\]

After prediction, values can be converted back to the original price scale using:

\[
\widehat{\text{price}}=\exp(\widehat{\text{log\_price}})-1
\]

This transformation can help models handle the skewed distribution of listing prices.

## 7. Final Data-Quality Validation

The final feature matrices and target arrays were checked for missing values, infinite values, and row-alignment problems.

| Validation check | Result |
|---|---:|
| Missing values in `X_train` | 0 |
| Missing values in `X_valid` | 0 |
| Infinite values in `X_train` | 0 |
| Infinite values in `X_valid` | 0 |
| Missing values in `y_train` | 0 |
| Missing values in `y_valid` | 0 |
| Training feature-target alignment | Passed |
| Validation feature-target alignment | Passed |

The results confirm that the final datasets contain no missing or infinite values and that feature rows align with their corresponding target values.

## 8. Saved Artifacts

The following preprocessing artifacts have been saved in the `data` directory:

| Artifact | Purpose |
|---|---|
| `preprocessor.joblib` | Existing structured-data preprocessing and encoding |
| `text_pipeline.joblib` | Fitted TF-IDF vectorizers and TruncatedSVD models |
| `train_processed.csv` | Processed training records |
| `valid_processed.csv` | Processed validation records |

The saved text pipeline contains the fitted product-name vectorizer, description vectorizer, product-name SVD model, and description SVD model.

The final combined feature matrices, `X_train` and `X_valid`, currently exist in the notebook session. They have not yet been saved as standalone feature datasets.

## 9. Conclusion

The preprocessing and feature-engineering workflow has produced clean training and validation feature matrices containing 212 features per listing. Structured listing attributes are combined with reduced text representations generated from product names and descriptions.

Dataset coverage, split exclusivity, feature dimensions, missing values, infinite values, and feature-target alignment have been verified.

The resulting datasets are ready for regression modelling. The next stage is to establish a baseline model, evaluate its performance on the validation dataset, compare alternative regression algorithms, and tune promising models using consistent evaluation metrics.

In [22]:
import os
import joblib

os.makedirs("../data", exist_ok=True)

joblib.dump(
    {
        "X_train": X_train,
        "X_valid": X_valid,
        "y_train": y_train,
        "y_valid": y_valid,
    },
    "../data/model_features.joblib",
    compress=3,
)

print("Saved model-ready datasets.")
print("Path: ../data/model_features.joblib")

Saved model-ready datasets.
Path: ../data/model_features.joblib


In [23]:
import os

path = "../data/model_features.joblib"

print("File exists:", os.path.exists(path))
print("File size (MB):", round(os.path.getsize(path) / (1024 ** 2), 2))

File exists: True
File size (MB): 1047.15


## 10. Overall Conclusion

This notebook completed the preprocessing and feature-engineering stage of the Mercari Price Suggestion Challenge.

The original dataset was validated, and listings with non-positive prices were removed. The remaining 1,481,661 records were divided into training and validation datasets, with no overlap or missing eligible records.

Structured features were engineered from listing attributes, including item condition, shipping information, brand, hierarchical categories, and text length statistics. Product names and item descriptions were converted into numerical representations using TF-IDF. TruncatedSVD reduced the resulting 30,000 text dimensions to 200 components per listing.

These text components were combined with 12 structured features to produce model-ready datasets containing 212 features per record.

Final validation confirmed that the feature matrices and target arrays contained no missing or infinite values and that their dimensions were correctly aligned. The fitted structured-data preprocessor and text-processing models were saved as reusable artifacts. The combined feature matrices and targets were also saved in `model_features.joblib`.

The resulting datasets are ready for regression modelling. However, model performance has not yet been evaluated, and the effectiveness of the engineered features must be established through validation experiments.

### Deliverables

- `train_processed.csv` and `valid_processed.csv`: processed training and validation records.
- `preprocessor.joblib`: fitted structured-data preprocessing and encoding object.
- `text_pipeline.joblib`: fitted TF-IDF vectorizers and TruncatedSVD models.
- `model_features.joblib`: combined training and validation feature matrices and target arrays.

### Next Stage

The next notebook, **03_Modeling_Tuning.ipynb**, will establish a baseline regression model, evaluate its performance using appropriate metrics, compare alternative algorithms, and tune promising models.

The large generated feature file must be shared separately if it exceeds GitHub's file-size limits. The code, notebook, dependency information, and instructions for obtaining or regenerating the data should be maintained in the project repository to support reproducibility.